# PM2.5 and Lung Cancer Incidence Analysis in China (2015)

This study investigates the association between PM2.5 concentration and lung cancer incidence across 121 Chinese cities.

Spatial autocorrelation analysis (Global and Local Moran’s I) was conducted using ArcGIS, and the results are incorporated here for interpretation.

The analytical workflow includes:
- Spearman correlation analysis  
- Poisson regression for relative risk estimation  
- Population Attributable Fraction (PAF) calculation  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import spearmanr

plt.rcParams['font.family'] = ['DejaVu Sans']

## Data Loading and Preparation

The dataset has been preprocessed and includes PM2.5 concentration, lung cancer incidence rates, and population data.

In [ ]:
df = pd.read_csv("D:\Documents\GitHub\PM2.5_LungCancer_DataAnalysis\data\processed\整理后原始数据.xlsx")

df = df.rename(columns={
    'PM2.5 （微克每立方米）': 'pm25',
    '世标率(男)': 'male_rate',
    '世标率(女)': 'female_rate',
    '常住人口（男/万人）': 'male_pop_10k',
    '常住人口（女/万人）': 'female_pop_10k'
})

df['male_population'] = df['male_pop_10k'] * 10000
df['female_population'] = df['female_pop_10k'] * 10000
df['total_population'] = df['male_population'] + df['female_population']

df['male_cases'] = df['male_rate'] / 100000 * df['male_population']
df['female_cases'] = df['female_rate'] / 100000 * df['female_population']
df['total_cases'] = df['male_cases'] + df['female_cases']

df = df.dropna()
df.head()

## Descriptive Statistics

In [ ]:
df[['pm25','male_rate','female_rate']].describe()

## Spearman Correlation Analysis

In [ ]:
corr_male, p_male = spearmanr(df['pm25'], df['male_rate'])
corr_female, p_female = spearmanr(df['pm25'], df['female_rate'])

print("PM2.5 vs Male:", corr_male, p_male)
print("PM2.5 vs Female:", corr_female, p_female)

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.scatter(df['pm25'], df['male_rate'], alpha=0.6)
plt.title(f'Male (ρ={corr_male:.2f})')

plt.subplot(1,2,2)
plt.scatter(df['pm25'], df['female_rate'], alpha=0.6)
plt.title(f'Female (ρ={corr_female:.2f})')

plt.tight_layout()
plt.show()

## Poisson Regression Model

In [ ]:
def poisson_model(cases, population):
    X = sm.add_constant(df['pm25'])
    model = sm.GLM(cases, X,
                   family=sm.families.Poisson(),
                   offset=np.log(population)).fit()
    return model

model_total = poisson_model(df['total_cases'], df['total_population'])
model_male = poisson_model(df['male_cases'], df['male_population'])
model_female = poisson_model(df['female_cases'], df['female_population'])

In [ ]:
model_total.summary()

## Relative Risk (RR)

In [ ]:
def get_rr(model):
    beta = model.params['pm25']
    ci = model.conf_int().loc['pm25']
    
    rr10 = np.exp(10 * beta)
    rr10_ci = np.exp(10 * ci)
    
    return rr10, rr10_ci

rr_total = get_rr(model_total)
rr_male = get_rr(model_male)
rr_female = get_rr(model_female)

print("Total:", rr_total)
print("Male:", rr_male)
print("Female:", rr_female)

## Population Attributable Fraction (PAF)

In [ ]:
tmrel = 5.0
beta = np.log(1.09) / 10

def calc_rr(pm25):
    return np.exp(beta * (pm25 - tmrel))

def calc_paf(pop, cases):
    weights = pop / pop.sum()
    rr = calc_rr(df['pm25'])
    
    paf = np.sum(weights * (rr - 1)) / np.sum(weights * rr)
    avoid = cases.sum() * paf
    return paf, avoid

paf_total = calc_paf(df['total_population'], df['total_cases'])
paf_male = calc_paf(df['male_population'], df['male_cases'])
paf_female = calc_paf(df['female_population'], df['female_cases'])

print(paf_total, paf_male, paf_female)

In [ ]:
labels = ['Total','Male','Female']
values = [paf_total[0]*100, paf_male[0]*100, paf_female[0]*100]

plt.bar(labels, values)
plt.ylabel('PAF (%)')
plt.title('Population Attributable Fraction')
plt.show()

## Conclusion

- PM2.5 concentration shows a positive association with lung cancer incidence.  
- Poisson regression indicates increased relative risk with higher exposure.  
- PAF results suggest a non-negligible proportion of cases could be avoided by reducing PM2.5 levels.  
- Spatial clustering (Moran’s I) is consistent with the observed statistical relationships.  